# Final submission: rebuild and check

This notebook rebuilds our best upload (upload 27, public macro-F1 0.5556681626467764) from the raw
competition files and checks that the result is byte for byte the uploaded file.

It runs the same command as the README:

```
python -m grenuke.build --config configs/u9_p4_n1x_order_plus_edge_borders_burst.toml --out final.csv
```

**Before you run it**

- Start Jupyter in the repository root, in the environment from the README's Setup section
  (CPython 3.13, an NVIDIA GPU).
- Put `train.csv`, `test.csv`, `accounts.csv`, `merchants.csv` and `sample_submission.csv` in
  `data/upi-fraud-rings/`, or set `GRENUKE_DATA_DIR` to the folder that holds `upi-fraud-rings/`.
- A first run takes about half an hour on one RTX 5060 Ti 16 GB. Every step is cached, so a second
  run takes seconds.

The outputs stored below are from a run on one RTX 5060 Ti with the cache already filled.

In [1]:
import hashlib
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

REPO = Path.cwd()
assert (REPO / "configs").is_dir() and (REPO / "src" / "grenuke").is_dir(), "start Jupyter in the repository root"

# The folder that holds upi-fraud-rings/ (the competition CSVs), and scratch space for cached steps.
os.environ.setdefault("GRENUKE_DATA_DIR", str(REPO / "data"))
os.environ.setdefault("GRENUKE_CACHE_DIR", str(REPO / "cache"))

CONFIG = "configs/u9_p4_n1x_order_plus_edge_borders_burst.toml"
OUT = "final.csv"
EXPECTED_SHA256 = "a6f3f01ba1081fd07d5e3809781c820c6df9a78ea6677acbe13542e3ff89804f"
DEVICE = "cuda"  # the GPU path is the one that rebuilds the file byte for byte (see Reproducibility in the README)

files = ("train.csv", "test.csv", "accounts.csv", "merchants.csv", "sample_submission.csv")
folder = Path(os.environ["GRENUKE_DATA_DIR"]) / "upi-fraud-rings"
missing = [name for name in files if not (folder / name).is_file()]
assert not missing, f"missing competition files: {missing} (see Setup in the README)"
print("competition files found:", ", ".join(files))

competition files found: train.csv, test.csv, accounts.csv, merchants.csv, sample_submission.csv


## 1. Install

Skip this cell if the environment from the README's Setup section is already active.

In [ ]:
# Optional: the pinned packages and this package, as in the README's Setup section.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True, cwd=REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True, cwd=REPO)

## 2. Build

Ingests the CSVs, builds the features, trains the models, decodes, applies the post rules and
writes `final.csv`.

In [2]:
env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(REPO / "src"), env.get("PYTHONPATH", "")]))
command = [sys.executable, "-m", "grenuke.build", "--config", CONFIG, "--out", OUT, "--device", DEVICE]
print("python", " ".join(command[1:]), flush=True)
with subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as build:
    for line in build.stdout:
        print(line, end="")
assert build.returncode == 0, "the build failed: see the output above"

python -m grenuke.build --config configs/u9_p4_n1x_order_plus_edge_borders_burst.toml --out final.csv --device cuda
== u9_p4_n1x_order_plus_edge_borders_burst: device cuda, 6 threads
== ingest
== features and matrix
== pool columns
== cross-fitted fraud score with pools
== decode
   template rule re-typed 125 rows
   post rule: 6 legit rows called phishing_collect
   n1x rule: 7 legit rows called phishing_collect (roles from all rows)
   block rule order: 7 fraud rows re-typed
   block rule mulebot: 11 fraud rows re-typed
   edge velocity_bot|layering_chain (min p 0.6): phone 473828 tag - rows 1 called velocity_bot: P(right of the border) 0.0370 (flat prior 0.0931)
   edge velocity_bot|layering_chain (min p 0.6): phone 473829 tag - rows 1 called velocity_bot: P(right of the border) 0.0532 (flat prior 0.1157)
   edge velocity_bot|layering_chain (min p 0.6): phone 473830 tag - rows 1 called velocity_bot: P(right of the border) 0.0765 (flat prior 0.1452)
   edge velocity_bot|layering_chai

## 3. Check

The sha256 of the rebuilt file against the uploaded file's, then the number of rows per class.

In [3]:
path = REPO / OUT
sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
with path.open() as handle:
    header = handle.readline().strip()
    labels = Counter(line.strip().rsplit(",", 1)[1] for line in handle if line.strip())
print("columns ", header)
print("rows    ", f"{sum(labels.values()):,}")
print("sha256  ", sha256)
print("expected", EXPECTED_SHA256)
print("RESULT  ", "IDENTICAL to the uploaded file" if sha256 == EXPECTED_SHA256 else "DIFFERS from the uploaded file")

columns  id,label
rows     260,582
sha256   a6f3f01ba1081fd07d5e3809781c820c6df9a78ea6677acbe13542e3ff89804f
expected a6f3f01ba1081fd07d5e3809781c820c6df9a78ea6677acbe13542e3ff89804f
RESULT   IDENTICAL to the uploaded file


In [4]:
for label, count in labels.most_common():
    print(f"{label:<24}{count:>9,}")

legit                     256,595
mule_account                  772
layering_chain                556
qr_swap                       538
phishing_collect              516
sim_swap                      367
velocity_bot                  334
refund_abuse                  279
account_takeover              233
fake_cashback                 207
loan_app_extortion            102
merchant_collusion             83
